# AF 8 -- dynamic ODLS (KERNEL_T = 3), fine-tuned from the AF 4 dynamic model

Runs on a **collaborator account** using the `odls_dynamic` folder shared by the project
owner (Editor access). Estimated training time: ~2.5 h on a free T4.

**Before you start:** accept the Drive share invite email, then
**Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.

Safe to re-run after a disconnect: training resumes from the exact batch where it stopped.
When you stop training yourself, wait for "Drive flushed -- everything is saved".

## 0. Connect the shared folder to this account's My Drive (automatic, once)
Finds the `odls_dynamic` folder shared with you and adds a shortcut to it at the top of
My Drive, so the notebook sees it at `/content/drive/MyDrive/odls_dynamic`.
Accept the Google sign-in pop-up.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
svc = build("drive", "v3")
me = svc.about().get(fields="user(emailAddress)").execute()["user"]["emailAddress"]
print("signed in as:", me)
FOLDER = "application/vnd.google-apps.folder"
shared = svc.files().list(
    q=f"name = 'odls_dynamic' and mimeType = '{FOLDER}' and sharedWithMe and trashed = false",
    fields="files(id, owners(emailAddress))").execute()["files"]
if not shared:
    raise RuntimeError(f"No folder 'odls_dynamic' is shared with {me}. Either the owner hasn't shared it "
                       f"with this address yet (or only just did -- wait a minute and re-run), or the wrong "
                       f"account was picked in the sign-in pop-up: run  !rm -rf /content/.config  and "
                       f"re-run this cell choosing the collaborator account.")
target = shared[0]["id"]
mine = svc.files().list(q="name = 'odls_dynamic' and 'root' in parents and trashed = false",
                        fields="files(id, mimeType, shortcutDetails)").execute()["files"]
if any(f.get("shortcutDetails", {}).get("targetId") == target for f in mine):
    print("shortcut already in My Drive -- nothing to do")
elif mine:
    raise RuntimeError("My Drive already has a different item called 'odls_dynamic' -- rename or "
                       "remove it, then run this cell again.")
else:
    svc.files().create(body={"name": "odls_dynamic", "mimeType": "application/vnd.google-apps.shortcut",
                             "parents": ["root"], "shortcutDetails": {"targetId": target}},
                       fields="id").execute()
    print(f"linked the shared folder (owner: {shared[0]['owners'][0]['emailAddress']}) into My Drive")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config -- the only cell you normally edit

In [ ]:
# Everything persistent goes under here (Google Drive).
DRIVE_ROOT = "/content/drive/MyDrive/odls_dynamic"
DRIVE_DATA_DIR = f"{DRIVE_ROOT}/ocmr_preprocessed"   # written once, ~4.8 GB (hard cap below)
MAX_DATA_GB = 8                                      # preprocessing stops before exceeding this

# Temporary, wiped when the runtime resets -- that's fine.
LOCAL_DATA_DIR = "/content/ocmr_preprocessed"        # fast local copy used for training
RAW_TMP_DIR = "/content/ocmr_tmp"                    # holds one raw OCMR file at a time

REPO_URL = "https://github.com/Shambhawi419/1D_MRI.git"
REPO_DIR = "/content/1D_MRI"
CODE_DIR = f"{REPO_DIR}/odls_dynamic"

# Experiment. A new RUN_NAME = a new, separate run folder; the same RUN_NAME = resume.
AF = 8                     # preset for this notebook
KERNEL_T = 3               # preset for this notebook
SPARSE_DOMAIN = "yt"       # "yt" (paper) or "yf" (temporal-Fourier sparsity, k-t style)
RUN_NAME = f"dyn_af{AF:g}_kt{KERNEL_T}_{SPARSE_DOMAIN}_v2"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/runs/{RUN_NAME}"

EPOCHS = 20                # fine-tuning; raise and re-run Train if val is still improving
BATCH_SIZE = 32            # rows of a median-size slice; scaled per slice to keep GPU memory flat
TIME_LIMIT_MIN = 180       # stop cleanly + flush Drive every 3 h; re-run Train to continue
EXTRA_TRAIN_ARGS = "--init-checkpoint /content/drive/MyDrive/odls_dynamic/runs/dyn_af4_kt3_yt_v2/best.pt --lr 0.0005"

In [ ]:
import os, subprocess, sys
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
sys.path.insert(0, CODE_DIR)

from google.colab import drive


def ensure_drive():
    # If Drive is not mounted, /content/drive/... is just an empty LOCAL folder: checkpoints
    # written there vanish with the runtime and best.pt "doesn't exist". Train and Evaluate
    # call this first, so they always work on the real Drive.
    if not os.path.ismount('/content/drive'):
        print("Google Drive is not connected -- reconnecting (accept the pop-up if shown) ...")
        drive.mount('/content/drive', force_remount=True)
    if not os.path.isdir(DRIVE_DATA_DIR):
        raise RuntimeError(f"{DRIVE_DATA_DIR} not found on Drive -- check DRIVE_ROOT / the mounted account")


def flush_drive():
    # Blocks until everything written to Drive has finished uploading. Colab uploads
    # in the background; files still queued are LOST if the runtime is killed (e.g. when
    # the free GPU quota runs out), so Train and Evaluate call this when they end.
    # Don't interrupt it -- let it print "Drive flushed".
    print("flushing Google Drive (waiting for uploads to finish -- don't stop this cell) ...")
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
    print("Drive flushed -- everything is saved and Drive is connected again.")


!nvidia-smi --query-gpu=name,memory.total --format=csv

## Optional: self-test (1-2 min, no data needed)
Run after pulling new code, before spending GPU time.

In [ ]:
!cd {CODE_DIR} && python -u selftest.py

## 1. Check the shared data (preprocessing was done once by the owner -- not re-run here)
Read-only check that the 128 preprocessed slices are visible through the shared folder.

In [ ]:
import csv
rows = list(csv.DictReader(open(f"{DRIVE_DATA_DIR}/manifest.csv")))
n_files = len([n for n in os.listdir(f"{DRIVE_DATA_DIR}/slices") if n.endswith(".npy")])
print(f"manifest: {len(rows)} slices, {n_files} slice files on Drive")
assert len(rows) == 128 and n_files == 128, "shared data incomplete -- tell the owner"

## 2. Copy the preprocessed data to local disk (every new session, ~1-2 min)
Training reads many small row slices; local disk is much faster than Drive for that.

If Google Drive's connection drops mid-copy ("Transport endpoint is not connected"),
this cell reconnects Drive and carries on by itself; files already copied are skipped.
Check that it ends with **128 slices available locally** before training (fewer means
preprocessing hasn't finished -- re-run step 1).

In [ ]:
import shutil, time
from google.colab import drive


def remount():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)


src_dir, dst_dir = f"{DRIVE_DATA_DIR}/slices", f"{LOCAL_DATA_DIR}/slices"
os.makedirs(dst_dir, exist_ok=True)
for attempt in range(1, 6):
    try:
        names = sorted(n for n in os.listdir(src_dir) if n.endswith(".npy"))
        copied = 0
        for name in names:
            src, dst = f"{src_dir}/{name}", f"{dst_dir}/{name}"
            if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
                continue                          # already copied
            shutil.copy(src, dst + ".tmp")        # copy to a temp name, then rename,
            os.replace(dst + ".tmp", dst)         # so a drop never leaves a half-copied file
            copied += 1
        for name in ("manifest.csv", "split.csv"):
            shutil.copy(f"{DRIVE_DATA_DIR}/{name}", f"{LOCAL_DATA_DIR}/{name}")
        print(f"done: copied {copied} new files, {len(names)} slices available locally")
        break
    except OSError as e:
        print(f"Drive connection dropped ({e}); reconnecting, attempt {attempt}/5 ...")
        time.sleep(5)
        remount()
else:
    raise RuntimeError("Drive kept disconnecting -- restart the runtime and run this cell again")

## 3. Train
Re-running this cell with the same `RUN_NAME` resumes exactly where it stopped
(model, optimizer, scheduler, AMP scaler, RNG, epoch and batch position).
Checkpoints: every 200 steps and every epoch, to `CHECKPOINT_DIR` on Drive.
`history.csv` there has per-epoch losses and every phase's eta1 / eta2 / lambda.

**Drive safety:** Colab uploads Drive files in the background. If the runtime is killed
before an upload finishes (e.g. the free GPU quota runs out), that file is lost. So this
cell stops training cleanly after `TIME_LIMIT_MIN` and then **flushes Drive** (waits until
every checkpoint is uploaded) -- also if you stop the cell yourself. **Let the flush finish**
(wait for "Drive flushed"; accept the Drive pop-up if one appears). Re-run the cell to continue.

In [ ]:
train_cmd = (f"cd {CODE_DIR} && python -u train.py --data-dir {LOCAL_DATA_DIR} "
             f"--checkpoint-dir {CHECKPOINT_DIR} --af {AF} --kernel-t {KERNEL_T} "
             f"--sparse-domain {SPARSE_DOMAIN} --epochs {EPOCHS} --batch-size {BATCH_SIZE} "
             f"--time-limit-min {TIME_LIMIT_MIN} --num-workers 2 {EXTRA_TRAIN_ARGS}")
ensure_drive()
try:
    get_ipython().system(train_cmd)
finally:
    flush_drive()   # runs however training ends: finished, time limit, error, or you pressed stop

## 4. Evaluate on the held-out test split
The model is rebuilt from `best.pt` alone (no architecture flags). Prints the model
and the zero-filled baseline side by side; writes
`CHECKPOINT_DIR/eval_test_af{AF}/results.json` plus a few example reconstructions.

In [ ]:
ensure_drive()
!cd {CODE_DIR} && python -u evaluate.py --data-dir {LOCAL_DATA_DIR} --checkpoint {CHECKPOINT_DIR}/best.pt --split test
flush_drive()

## 5. Look at a reconstruction

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
ex = sorted(glob.glob(f"{CHECKPOINT_DIR}/eval_test_af{AF:g}/*.npz"))[0]
d = np.load(ex)
ref, zf, rec = (d[k].astype(float) for k in ("ref", "zero_filled", "odls_dynamic"))
t, col = ref.shape[0] // 2, ref.shape[2] // 2
vmax = ref.max()
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
panels = [(ref, "fully sampled"), (zf, f"zero-filled AF={AF}"), (rec, "dynamic ODLS" if KERNEL_T > 1 else "frame-by-frame ODLS"), (np.abs(rec - ref) * 5, "|error| x5")]
for j, (img, title) in enumerate(panels):
    ax[0, j].imshow(img[t], cmap="gray", vmin=0, vmax=vmax); ax[0, j].set_title(f"{title}, frame {t}")
    ax[1, j].imshow(img[:, :, col].T, cmap="gray", vmin=0, vmax=vmax, aspect="auto"); ax[1, j].set_title("y-t profile")
for a in ax.ravel():
    a.axis("off")
plt.suptitle(os.path.basename(ex)); plt.tight_layout(); plt.show()

## 6. Compare runs
Collects every `results.json` under `DRIVE_ROOT/runs`. The key comparison is
dynamic (`KERNEL_T=3`) vs frame-by-frame (`KERNEL_T=1`, same pipeline) vs zero-filled:
the gap between the first two is what using the time dimension buys.

To create the static baseline: set `KERNEL_T = 1` in the Config cell, re-run it, then
re-run cells 3 and 4 (it gets its own `RUN_NAME` folder automatically).

In [ ]:
import json
for path in sorted(glob.glob(f"{DRIVE_ROOT}/runs/*/eval_*/results.json")):
    r = json.load(open(path))
    run = path.split("/runs/")[1].split("/")[0]
    for m, o in r["overall"].items():
        print(f"{run:28s} {r['split']:5s} AF{r['af_requested']:<4g} {m:13s} "
              f"RLNE {o['RLNE_mean'] * 100:6.2f}  PSNR {o['PSNR_mean']:6.2f}  SSIM {o['SSIM_mean'] * 100:6.2f}")

## Other workflows from saved checkpoints
- **Fine-tune at another AF** (e.g. AF 8 from the AF 4 model): set `AF = 8` in Config
  (new `RUN_NAME`), and set `EXTRA_TRAIN_ARGS = "--init-checkpoint /content/drive/MyDrive/odls_dynamic/runs/dyn_af4_kt3_yt_v2/best.pt"`.
- **Train longer**: raise `EPOCHS` and re-run cell 3 -- it continues the same run.
- **Use a model in your own code**:
  `from checkpoint_utils import build_model_from_checkpoint; model, ckpt = build_model_from_checkpoint(path, "cuda")`
- **Different split** (e.g. the 0.55T mismatch test):
  `!cd {CODE_DIR} && python -u preprocess_ocmr.py --out-dir {DRIVE_DATA_DIR} --split-mode holdout-scanner --resplit-only`,
  then re-run cell 2 and train with a new `RUN_NAME`.